#### Steps taken:
1. Read bronze drivers data.
2. Keep only columns required for analytics, will drop the url column.
3. Standardize the column name using snake_case (ex: driverID -> driver_id, dateOfbirth -> date_of_birth)
4. Concatenate given name & family name to create a new column called driver_name and convert that value to Title case
5. Remove duplicate records.
6. Transform values of columns to title case (ex: nationality).
7. Write the transform data to silver driver table.

In [0]:
%run ../environment_config

In [0]:
bronze_table = f"{catalog}.{bronze_schema}.drivers"
silver_table = f"{catalog}.{silver_schema}.drivers"

In [0]:
drivers_df = spark.read.table(bronze_table)

In [0]:
drivers_selected_df = drivers_df.drop("url")

In [0]:
drivers_renamed_df = drivers_selected_df.withColumnsRenamed({
                            "driverId" : "driver_id",
                            "dateOfBirth" : "date_of_birth"
})

In [0]:
drivers_concat_df = drivers_renamed_df.withColumn( 
                                "driver_name", F.initcap(F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName")))
).drop(F.col("name"))

In [0]:
driver_distinct_df = drivers_concat_df.dropDuplicates(["driver_id"])

In [0]:
drivers_final_df = driver_distinct_df.withColumn("nationality", F.initcap(F.col("nationality")))

In [0]:
display(drivers_final_df)

In [0]:
(
    drivers_final_df.write
                    .format("delta")
                    .mode("overwrite")
                    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))